# 01.6 The ML Project Lifecycle and Where Projects Actually Die

> **Prerequisites:** 01.1 (baseline, capacity, drift) · 01.2 (data contract) ·
> 01.3 (manifests, noise bands) · 01.4 (serving contract) · 01.5 (label spec, economics)
> **What you'll learn:**
> - Walk the lifecycle as an instrumented pipeline whose every stage emits a checkable artifact
> - Write the gate suite that guards each stage transition, and run it as one harness
> - Distinguish a defect a gate can catch from a mis-specification no gate can catch
> - Measure detection distance: which defects reach production because nothing offline sees them
> - Turn "should we retrain?" into a function with thresholds instead of a recurring meeting
> **Level:** Beginner · **Series:** 01 The ML Landscape & Project Lifecycle

> ⚡ **Monday 2026-06-01, 10:15** — the late-payment model is decommissioned after fourteen
> months, having never demonstrably paid for itself. Every gate in its CI suite was green on the
> day it shipped and green on the day it was switched off. The cause: the two defects that
> mattered were introduced before a line of model code existed, and no check the team built could
> see either one.

## Concept
### Plain-English Explanation

The five preceding notebooks each solved one problem and each ended with a rule: measure the
baseline, contract the data, record the manifest, honour the serving contract, derive the
break-even precision. Assembled, those rules are a lifecycle — frame, label, data, baseline,
model, evaluate, ship, monitor — and it is tempting to draw it as a diagram with arrows and
consider the subject covered.

The diagram is not the useful part. What matters is that each stage *constrains* every stage
after it, and that a mistake made early is discovered late, if at all. A wrong join is caught in
minutes by a data contract. A wrong question is caught in quarters by a business that stops
funding the project, because nothing in a test suite has an opinion about whether you asked the
right thing.

So rather than survey the stages, this notebook builds the pipeline as an instrumented object,
writes a gate for every transition, and then deliberately breaks it — seven ways — to see which
gates actually fire. Five of the seven defects are caught offline. Two reach production, and
those two are the ones that kill projects.

### Technical Explanation

A lifecycle stage is worth naming only if it produces an **artifact** that a later stage consumes
and a gate can inspect. On that criterion the stages are: *frame* produces a problem statement
and a split strategy; *label* produces a target column and its spec; *data* produces the
modelling table; *baseline* produces a floor to beat; *model* produces a fitted artifact;
*evaluate* produces a metric with a noise band; *ship* produces a serving contract; *monitor*
produces a calibration signal and a retrain decision.

The gate suite guarding those transitions is assembled from the earlier notebooks rather than
invented here: the data contract from 01.2, a leakage blocklist for post-outcome columns, a
temporal-split assertion, the baseline comparison from 01.1, the noise band from 01.3, the
serving contract from 01.4, and the calibration monitor that 01.1's incident demanded. Seven
checks, each cheap, each mechanical.

⭐ **CRITICAL CONCEPT** — those gates test whether the pipeline did what you asked. Not one of
them tests whether you asked for the right thing. That distinction separates a **defect**, where
implementation diverges from intent, from a **mis-specification**, where intent itself is wrong.
Defects are catchable by machinery. Mis-specifications are catchable only by a conversation held
early and written down, and their signature is that every automated check stays green while the
project fails.

The measurement that makes this concrete is **detection distance**: the gap between the stage
where a defect enters and the earliest gate that can see it. Where that gap is zero the defect is
a bug and costs an afternoon. Where no gate can see it, the remaining detector is a customer, a
finance review, or fourteen months of funding.

### Mental Model

Every gate you can write asks "did the pipeline do what I asked?". Nothing in CI asks "did I ask
for the right thing?" — that check is a conversation, held before any code exists, and written
down where the next engineer will find it.

## How It Works

```text
  STAGE       ARTIFACT PRODUCED               GATE GUARDING THE TRANSITION      FROM
  ---------------------------------------------------------------------------------------
  frame    -> problem statement, split      G3 temporal split assertion         01.4/01.5
  label    -> target column + label spec    G2 leakage blocklist                01.5
  data     -> modelling table               G1 data contract (grain, dtypes)    01.2
  baseline -> the floor to beat             G4 model must beat the rule         01.1
  model    -> fitted artifact               (manifest hash recorded)            01.3
  evaluate -> metric + noise band           G5 delta must clear the band        01.3
  ship     -> serving contract              G6 queue size == capacity           01.4
  monitor  -> calibration signal            G7 predicted rate ~ actual rate     01.1

  Defects a gate can see (implementation diverged from intent):
      duplicate rows . leaked column . random split . threshold selection . no noise band

  Defects no gate can see (intent itself was wrong):
      wrong horizon      - predicts 'late by 7 days' when collections escalates at 30
      survivorship       - trains on resolved invoices only, scores every issued invoice
```

Two mechanisms sit behind that table.

**Gates are assertions about artifacts, so they can only see what an artifact records.** The
serving-contract gate compares queue size to capacity because both are numbers in the run. No
gate compares the label's horizon to the horizon the collections team escalates on, because the
second number exists only in somebody's head — and a check cannot be written against a fact that
was never written down. This is why the label spec of 01.5 and the manifest config block of 01.3
matter more than they first appear: writing a decision down is what converts it from unfalsifiable
into checkable.

**The population a model is trained on is defined by a join, and joins are silent.** The modelling
table this series has used throughout comes from an inner join between invoices and payments,
which 01.2 measured as dropping 12,529 unpaid invoices. Every number reported in five notebooks
was computed on invoices that were eventually paid. Production, however, scores invoices at issue
time, including the ones that will never be paid at all. Nothing in the gate suite notices,
because the training table is internally consistent and the metrics computed on it are correct —
they are simply correct about a population that is not the one being served.

## Hands-On Build
### Stage A — from scratch

Run the lifecycle once, cleanly, and make every stage report the artifact it produced. This is
the object the gates will inspect; without it, "the lifecycle" is a diagram rather than a thing.

In [1]:
import importlib.util
import sys
from pathlib import Path

import numpy as np
import pandas as pd

LAB = Path.cwd() / "_lab" / "lab_01.6_lifecycle.py"
spec = importlib.util.spec_from_file_location("lab_01_6", LAB)
lab = importlib.util.module_from_spec(spec)
sys.modules["lab_01_6"] = lab
spec.loader.exec_module(lab)

raw = pd.read_csv(lab.lab11.RAW / "invoices.csv.gz")
df_dedup, _ = lab.lab11.build_dataset()
dupes = df_dedup.sample(n=int(raw.duplicated().sum()), random_state=lab.SEED)
df_raw = pd.concat([df_dedup, dupes], ignore_index=True)   # a "no dedup" universe

clean = lab.build_run(df_dedup, df_raw, set())             # walk every stage once
print(f"frame     temporal split, {clean.n_train:,} training rows")
print(f"label     late = paid > 7 days past due; base rate {clean.actual_rate:.4f}")
print(f"baseline  dunning rule precision {clean.baseline_precision:.4f}")
print(f"model     logistic regression precision {clean.precision:.4f}")
print(f"evaluate  delta {clean.precision - clean.baseline_precision:+.4f}, "
      f"noise band {clean.noise_band:.4f}")
print(f"ship      queue {clean.queue_size:,} against capacity {clean.capacity:,}")
print(f"monitor   predicted {clean.predicted_rate:.3f} vs actual {clean.actual_rate:.3f}")
print(f"manifest  {clean.manifest_hash}")

frame     temporal split, 60,000 training rows
label     late = paid > 7 days past due; base rate 0.2632
baseline  dunning rule precision 0.4254
model     logistic regression precision 0.4499
evaluate  delta +0.0245, noise band 0.0031
ship      queue 7,095 against capacity 7,095
monitor   predicted 0.271 vs actual 0.263
manifest  89b5366dceef


Eight stages, eight artifacts, one manifest hash tying them together. Every number here has been
earned in an earlier notebook: the baseline of 0.4254 is 01.1's dunning rule, the noise band of
0.0031 is 01.3's bootstrap discipline, the queue of 7,095 against a capacity of 7,095 is 01.4's
serving contract, and the calibration comparison of 0.271 against 0.263 is the signal whose
absence caused 01.1's incident.

### Stage B — idiomatic

The gate suite is those checks written as functions over the run object, so that "is this ready
to ship?" becomes an executable question rather than a review meeting.

In [2]:
base_results = lab.report(clean, "gate suite on the clean run:")

  gate suite on the clean run:
    [PASS] G1 data contract         duplicate rows dropped
    [PASS] G2 leakage blocklist     no post-outcome features
    [PASS] G3 temporal split        split is temporal
    [PASS] G4 beats baseline        model 0.4499 vs rule 0.4254
    [PASS] G5 clears noise band     delta +0.0245 vs band 0.0031
    [PASS] G6 serving contract      queue 7,095 vs capacity 7,095
    [PASS] G7 calibration           predicted 0.271 vs actual 0.263


Seven green. This is what a healthy pipeline looks like, and it is exactly what the team in the
cold open saw on the day they shipped — which is the point of running it before breaking
anything.

### Stage C — production

A gate suite is only worth what it catches, and the way to find that out is to break the pipeline
on purpose. Each defect below swaps exactly one stage's decision for a defensible-looking wrong
one, and the suite runs unchanged.

In [3]:
matrix = {}
for key, (stage, description) in lab.DEFECTS.items():
    r = lab.build_run(df_dedup, df_raw, {key})
    caught = [g for g, ok in lab.report(r, f"defect at '{stage}': {description}").items()
              if not ok and base_results[g]]
    matrix[key] = caught
    print(f"  -> precision {r.precision:.4f}"
          + (f", production {r.prod_precision:.4f}" if r.prod_precision is not None else "")
          + f"  |  caught by: {', '.join(caught) if caught else 'NOTHING'}\n")

  defect at 'data': skip the duplicate-row drop (SPEC M1)
    [FAIL] G1 data contract         grain violated: duplicate invoice rows present
    [PASS] G2 leakage blocklist     no post-outcome features
    [PASS] G3 temporal split        split is temporal
    [PASS] G4 beats baseline        model 0.4465 vs rule 0.4249
    [PASS] G5 clears noise band     delta +0.0216 vs band 0.0024
    [PASS] G6 serving contract      queue 7,120 vs capacity 7,120
    [PASS] G7 calibration           predicted 0.271 vs actual 0.263
  -> precision 0.4465  |  caught by: G1 data contract



  defect at 'label': add reminder_count, written after the outcome (SPEC M10)
    [PASS] G1 data contract         duplicate rows dropped
    [FAIL] G2 leakage blocklist     post-outcome feature present: reminder_count
    [PASS] G3 temporal split        split is temporal
    [PASS] G4 beats baseline        model 0.7635 vs rule 0.4254
    [PASS] G5 clears noise band     delta +0.3381 vs band 0.0022
    [PASS] G6 serving contract      queue 7,095 vs capacity 7,095
    [PASS] G7 calibration           predicted 0.267 vs actual 0.263
  -> precision 0.7635, production 0.2956  |  caught by: G2 leakage blocklist



  defect at 'frame': split randomly on time-ordered invoices
    [PASS] G1 data contract         duplicate rows dropped
    [PASS] G2 leakage blocklist     no post-outcome features
    [FAIL] G3 temporal split        split is random on time-ordered data
    [PASS] G4 beats baseline        model 0.4430 vs rule 0.4318
    [PASS] G5 clears noise band     delta +0.0112 vs band 0.0045
    [PASS] G6 serving contract      queue 7,095 vs capacity 7,095
    [PASS] G7 calibration           predicted 0.264 vs actual 0.266
  -> precision 0.4430  |  caught by: G3 temporal split



  defect at 'ship': select the queue by score >= 0.5, not by capacity
    [PASS] G1 data contract         duplicate rows dropped
    [PASS] G2 leakage blocklist     no post-outcome features
    [PASS] G3 temporal split        split is temporal
    [PASS] G4 beats baseline        model 0.5743 vs rule 0.4254
    [PASS] G5 clears noise band     delta +0.1488 vs band 0.0031
    [FAIL] G6 serving contract      queue 1,973 vs capacity 7,095
    [PASS] G7 calibration           predicted 0.271 vs actual 0.263
  -> precision 0.5743  |  caught by: G6 serving contract



  defect at 'evaluate': report one run, with no variance estimate
    [PASS] G1 data contract         duplicate rows dropped
    [PASS] G2 leakage blocklist     no post-outcome features
    [PASS] G3 temporal split        split is temporal
    [PASS] G4 beats baseline        model 0.4499 vs rule 0.4254
    [FAIL] G5 clears noise band     no variance estimate was computed
    [PASS] G6 serving contract      queue 7,095 vs capacity 7,095
    [PASS] G7 calibration           predicted 0.271 vs actual 0.263
  -> precision 0.4499  |  caught by: G5 clears noise band



  defect at 'frame': predict 'late by 7 days' when collections escalates at 30
    [PASS] G1 data contract         duplicate rows dropped
    [PASS] G2 leakage blocklist     no post-outcome features
    [PASS] G3 temporal split        split is temporal
    [PASS] G4 beats baseline        model 0.4499 vs rule 0.4254
    [PASS] G5 clears noise band     delta +0.0245 vs band 0.0031
    [PASS] G6 serving contract      queue 7,095 vs capacity 7,095
    [PASS] G7 calibration           predicted 0.271 vs actual 0.263
  -> precision 0.4499  |  caught by: NOTHING



Five defects, five gates firing, each defect caught by the check written for it. That is a
correctly functioning suite and it is also the least interesting possible result — a suite that
catches the defects you thought of is table stakes.

⚠️ Note the fourth block. The leakage defect reports an offline precision of 0.7635 against the
clean run's 0.4499 while every gate except the blocklist stays green: it beats the baseline
enormously, clears the noise band by a hundredfold, and is well calibrated. Had the blocklist not
existed — had nobody written down that `reminder_count` is populated after the outcome — this
defect would have sailed through as the best result the team had ever produced.

The seventh block is the one to sit with: the wrong-horizon defect is caught by **nothing**.

## Evaluation

The object under evaluation is the gate suite, so the metric is which defects it catches, and the
harness is the injection matrix above plus the two mis-specifications below. The baseline is a
project with no gates at all, where every defect reaches production.

In [4]:
# The defect this series' own pipeline has: the modelling table is an inner join to
# payments, so every number reported in 01.1-01.5 describes invoices that were paid.
pay_ids = set(pd.read_csv(lab.lab11.RAW / "payments.csv.gz",
                          usecols=["invoice_id"])["invoice_id"])
win = raw.drop_duplicates().assign(issue_date=lambda d: pd.to_datetime(d["issue_date"]))
win = win.loc[(win["issue_date"] >= "2025-01-01") & (win["issue_date"] < "2025-06-01")]
never_paid = win.loc[~win["invoice_id"].isin(pay_ids)]
resolved_n = len(lab.lab11.windows(df_dedup)["test_stable"])
print(f"issued in the evaluation window : {len(win):>8,}")
print(f"resolved - all we ever measured : {resolved_n:>8,}")
print(f"never paid, never in training   : {len(never_paid):>8,}"
      f"   ({len(never_paid) / len(win):.1%} of production scoring volume)\n")
survivorship = lab.Run(defects={"survivorship"}, split_kind="temporal",
                       features=list(lab.lab11.NUM), dedup_applied=True,
                       n_train=clean.n_train, precision=clean.precision,
                       baseline_precision=clean.baseline_precision,
                       noise_band=clean.noise_band, queue_size=clean.queue_size,
                       capacity=clean.capacity, predicted_rate=clean.predicted_rate,
                       actual_rate=clean.actual_rate)
res = lab.report(survivorship, "gate suite on the survivorship defect:")
print(f"  caught by: {[g for g, ok in res.items() if not ok] or 'NOTHING'}")

issued in the evaluation window :   27,857
resolved - all we ever measured :   27,290
never paid, never in training   :      567   (2.0% of production scoring volume)

  gate suite on the survivorship defect:
    [PASS] G1 data contract         duplicate rows dropped
    [PASS] G2 leakage blocklist     no post-outcome features
    [PASS] G3 temporal split        split is temporal
    [PASS] G4 beats baseline        model 0.4499 vs rule 0.4254
    [PASS] G5 clears noise band     delta +0.0245 vs band 0.0031
    [PASS] G6 serving contract      queue 7,095 vs capacity 7,095
    [PASS] G7 calibration           predicted 0.271 vs actual 0.263
  caught by: NOTHING


Five of seven defects caught, two reaching production — and the two that get through are the two
that were never written down anywhere a check could reach.

The survivorship defect is real, present, and mine. Every metric in this series was computed on
the 27,290 invoices that were eventually paid, while 567 invoices issued in the same window — two
percent of what production would score — were never paid at all, are late by any definition, and
appear in no number anywhere in these six notebooks. The gate suite is green because the modelling
table is internally consistent; the table is simply about a different population from the one
being served. ⚠️ A test set drawn through the same join as the training set cannot detect this,
which is the general form of the problem and why series 12 treats population definition as part
of validation rather than as a data-prep detail.

The wrong-horizon defect is the same shape. The team measures precision 0.4499 against its own
seven-day label; collections escalates at thirty days, and against *that* question the identical
ranking scores 0.0519. The honest reading is not that the model is worthless — the thirty-day base
rate is 0.0175, so it still delivers 2.96x lift — but that its reported number describes a
different problem, and no gate compares a label's horizon to an operational trigger nobody
recorded.

In [5]:
leak = lab.build_run(df_dedup, df_raw, {"leakage"})
print(f"leakage      offline {leak.precision:.4f} vs clean {clean.precision:.4f}"
      f"  ({leak.precision - clean.precision:+.4f})")
print(f"             production, where the column is always 0: {leak.prod_precision:.4f}"
      f"  ({leak.prod_precision - clean.precision:+.4f} vs clean)")

# Random splits on temporal data: isolate the effect by holding the TEST SET fixed.
drift = lab.lab11.windows(df_dedup)["test_drift"]
held_out, contemporaneous = lab.train_test_split(drift, test_size=0.7,
                                                 random_state=lab.SEED)
pre_drift = lab.lab11.windows(df_dedup)["train"]
y_h = held_out["late"].to_numpy()
k_h = int(lab.lab11.dunning_rule(held_out).sum())
for tag, tr in [("honest (pre-drift rows only)", pre_drift.sample(n=lab.TRAIN_N,
                                                                 random_state=lab.SEED)),
                ("contaminated (+contemporaneous)",
                 pd.concat([pre_drift, contemporaneous]).sample(n=lab.TRAIN_N,
                                                                random_state=lab.SEED))]:
    sc = lab.lab11.fit_score(tr, held_out, seed=lab.SEED)
    p = lab.lab11.precision_recall_at_k(y_h, lab.lab11.topk_flag(sc, k_h))["precision"]
    print(f"{tag:<32} precision on identical test rows: {p:.4f}")

leakage      offline 0.7635 vs clean 0.4499  (+0.3136)
             production, where the column is always 0: 0.2956  (-0.1543 vs clean)


honest (pre-drift rows only)     precision on identical test rows: 0.5232


contaminated (+contemporaneous)  precision on identical test rows: 0.5280


Two calibrations of received wisdom, both worth having.

Leakage is as dangerous as its reputation and then some: the offline number improves by +0.3136
and the production number is *worse than the clean model*, at 0.2956 against 0.4499. A feature
that is populated after the outcome does not merely fail to help — it displaces the honest
features the model would otherwise have leaned on, so the model is left worse than if the feature
had never existed.

Random splitting on temporal data is *less* dangerous than its reputation, here. Holding the test
rows fixed and only varying whether training saw contemporaneous data, the precisions are 0.5232
and 0.5280 — effectively identical. That is consistent with what 01.1 found: this drift is a shift
in the base rate rather than in the relationship between features and outcome, so seeing the new
regime helps calibration and barely helps ranking. The textbook warning is real where the
relationship itself moves; asserting it without measuring it is how a team ends up with rituals
instead of reasons.

Finally, monitoring only earns its keep if it terminates in a decision, so the last stage is a
function rather than a meeting.

In [6]:
w = lab.lab11.windows(df_dedup)
for label, window in [("stable 2025-01..05", w["test_stable"]),
                      ("drift  2025-07..2026-05", w["test_drift"])]:
    train = w["train"].sample(n=lab.TRAIN_N, random_state=lab.SEED)
    scores = lab.lab11.fit_score(train, window, seed=lab.SEED)
    gap = abs(float(scores.mean()) - float(window["late"].mean()))
    k = int(lab.lab11.dunning_rule(window).sum())
    y = window["late"].to_numpy()
    rule_p = lab.lab11.precision_recall_at_k(y, lab.lab11.dunning_rule(window))["precision"]
    model_p = lab.lab11.precision_recall_at_k(y, lab.lab11.topk_flag(scores, k))["precision"]
    verdict = ("RETRAIN: calibration gap exceeds 0.05" if gap > 0.05 else
               "RETRAIN: model no longer beats the rule" if model_p <= rule_p else
               "hold: within tolerance")
    print(f"{label:<26} gap {gap:.3f}   model {model_p:.4f} vs rule {rule_p:.4f}"
          f"   -> {verdict}")

stable 2025-01..05         gap 0.008   model 0.4474 vs rule 0.4254   -> hold: within tolerance


drift  2025-07..2026-05    gap 0.123   model 0.5207 vs rule 0.5425   -> RETRAIN: calibration gap exceeds 0.05


The stable window holds at a calibration gap of 0.008; the drift window trips at 0.123 and would
have paged on the day the gateway migration landed rather than six weeks later. Note the second
clause too: in the drift window the model scores 0.5207 against the rule's 0.5425, so the baseline
has overtaken it — the condition 01.1 said should be watched forever, now expressed as code that
runs on a schedule.

## Design Patterns / Tradeoffs

**Offline gates versus staged rollout.** Offline gates are cheap, fast and deterministic: they run
in CI in seconds, block before anything reaches a customer, and give an unambiguous reason for
failure. Their limit is exactly what this notebook measures — they can only assert over recorded
artifacts, so they catch defects and are blind to mis-specifications. A staged rollout, where the
model serves a small share of traffic beside the incumbent with the business metric measured on
both, catches the other class: a wrong horizon or a survivorship-biased population shows up as a
business result that fails to move even though every model metric is healthy. It costs real time,
requires enough traffic for the comparison to be conclusive, and exposes some customers to the new
behaviour. Run both: gates block the defects before deployment, the rollout catches the questions
nobody validated. Where traffic is too small for a rollout to conclude, substitute a written
pre-registration of the expected business effect and check it against outcomes at a fixed date.

**Heavy process versus light process.** Every gate costs authoring time, run time and a
maintenance obligation, and a suite that is slow or flaky gets bypassed, which is worse than not
having it. Light process ships faster and relies on the team's attention, which works while the
team is small and the person who framed the problem is the person who deploys it. The tipping
point is handover: the moment somebody who was not in the framing conversation can change the
pipeline, undocumented intent becomes unenforceable, and the cost of a gate is repaid the first
time it fires. Add gates in the order of what has already gone wrong — every check in this suite
exists because an earlier notebook's incident demanded it — rather than importing a checklist
wholesale.

**Recommendation for PayFlow:** keep the seven gates in CI, add the two written artifacts that
would make the invisible defects visible — a label spec naming the operational trigger, and a
scoring-population definition naming who is eligible — and require a pre-registered business
metric with a review date for anything that ships. Those three additions cost an afternoon and
address precisely the failures the gates cannot.

## Production Scenario
### Symptoms

**Monday 2026-06-01, 10:15.** The late-payment model is decommissioned after fourteen months. The
post-mortem convenes because nobody can explain why a technically healthy system never produced a
measurable business result.

- The CI suite has been **green for its entire life**, including on the final day. Data contract,
  leakage blocklist, split assertion, baseline comparison, noise band, serving contract and
  calibration all pass.
- Offline precision at ship time was 0.4499 against a rule baseline of 0.4254, a delta of +0.0245
  against a noise band of 0.0031 — a real, reproducible improvement by every standard 01.3 set.
- Collections reports the queue was "usually fine" but that the invoices they were escalating were
  not the ones that ended up in serious arrears.
- Finance can find no quarter in which recovery improved relative to the pre-model trend, and the
  original business case was never revisited after launch.
- A junior engineer, asked to reproduce the launch numbers, does so exactly — the manifest hash
  matches — and reports that everything checks out.

In [7]:
wrong = lab.build_run(df_dedup, df_raw, {"wrong_horizon"})
stable = lab.lab11.windows(df_dedup)["test_stable"]
base30 = float((stable["days_late"] > 30).mean())
print(f"measured against its own 7-day label   : {wrong.precision:.4f}")
print(f"measured against the 30-day escalation : {wrong.business_precision:.4f}")
print(f"30-day base rate {base30:.4f}, so lift on the question that matters is "
      f"{wrong.business_precision / base30:.2f}x")
lab.report(wrong, "\ngate suite on the wrong-horizon defect:")

measured against its own 7-day label   : 0.4499
measured against the 30-day escalation : 0.0519
30-day base rate 0.0175, so lift on the question that matters is 2.96x
  
gate suite on the wrong-horizon defect:
    [PASS] G1 data contract         duplicate rows dropped
    [PASS] G2 leakage blocklist     no post-outcome features
    [PASS] G3 temporal split        split is temporal
    [PASS] G4 beats baseline        model 0.4499 vs rule 0.4254
    [PASS] G5 clears noise band     delta +0.0245 vs band 0.0031
    [PASS] G6 serving contract      queue 7,095 vs capacity 7,095
    [PASS] G7 calibration           predicted 0.271 vs actual 0.263


{'G1 data contract': True,
 'G2 leakage blocklist': True,
 'G3 temporal split': True,
 'G4 beats baseline': True,
 'G5 clears noise band': True,
 'G6 serving contract': True,
 'G7 calibration': True}

### Diagnosis

1. **Alert** — not an alert at all, but a funding review: fourteen months, no measurable effect.
   Candidate causes: the model is bad, the model is good but unused, the intervention does not
   work, or the model answers the wrong question.
2. **The standard observability ladder, walked first and entirely green.** Quality dashboards show
   precision stable and above baseline. Prediction logs show a sane score distribution. Input-data
   checks pass on schema, null rates and volumes. Drift analysis shows the post-migration shift
   already handled by the retraining schedule. The version diff is clean, and the manifest hash
   reproduces exactly. ⚠️ An all-green ladder beneath a failing project is not the absence of a
   finding — it *is* the finding, because it eliminates the entire class of implementation defects
   and leaves only hypotheses about what the system was asked to do.
3. **CI history** — green throughout, consistent with the ladder and with the reproduction. The
   pipeline demonstrably did what it was told, which also eliminates the reflex to hunt for a bug.
4. **Model-quality review** — offline precision 0.4499 against a 0.4254 baseline, delta well
   outside the 0.0031 noise band. The model is genuinely better than the rule at the task it was
   trained on, so "the model is bad" is eliminated.
5. **Interview with collections** — escalation happens at thirty days past due; the model predicts
   seven. Re-scoring the same ranking against the thirty-day label gives 0.0519 against a base rate
   of 0.0175 — real signal, 2.96x lift, but a fraction of what the team believed it was deploying,
   and never once measured in fourteen months.
6. **Population audit** — the modelling table is an inner join to payments. Of 27,857 invoices
   issued in a representative window, 27,290 were resolved and formed the entire basis of training
   and evaluation; 567 were never paid, are unambiguously the worst outcomes in the portfolio, and
   were structurally invisible to the model and to every metric reported about it.
7. **Gate-coverage review** — running the suite against both findings returns all green. There was
   never a check capable of detecting either, because neither the operational trigger nor the
   scoring population was written down anywhere a check could reference.

### Root Cause

Two mis-specifications, both introduced during framing and neither expressible as a gate. The
label used a seven-day threshold while the business escalates at thirty, so every reported metric
measured a different problem from the one being solved. And the population was defined by an inner
join to payments, so the model was trained and evaluated exclusively on invoices that were
eventually paid while production scored every issued invoice, including the two percent that never
would be.

### Fix

**Mitigation now.** Do not rebuild. Re-score the existing ranking against the thirty-day label to
establish what the model is actually worth on the real question — 2.96x lift is not nothing — and
give collections that number so the decommissioning decision is made on evidence rather than on
disappointment.

**Permanent fix.** Write the two missing artifacts. A label spec, per 01.5, naming the unit,
population, cutoff and horizon, with the horizon justified against collections' escalation
trigger. And a scoring-population definition naming exactly which invoices are eligible, so that
the never-paid rows are either included in training with an appropriate label or explicitly
excluded from scoring with a stated reason. Both go in the manifest config block from 01.3, which
makes them diffable, reviewable and — for the first time — checkable.

### Prevention

- **Add the two gates the artifacts make possible.** Once the operational trigger is recorded, a
  gate can assert that the label's horizon equals it. Once the scoring population is recorded, a
  gate can assert that the training population matches it. Neither check was writable before;
  both are trivial afterwards.
- **Pre-register the business metric.** Name the expected effect, the measurement and the review
  date before launch. Fourteen months passed without anyone being obliged to answer "did it work?"
- **Run a staged rollout with the incumbent as control.** The rule was available throughout and
  costs nothing to score in parallel; the comparison would have surfaced the gap within a quarter.
- **Interview the operator during framing, not during the post-mortem.** The thirty-day trigger was
  never secret. It was simply never asked for, and the entire project's value turned on it.
- **Treat an all-green suite as evidence about defects only.** Green means the pipeline did what
  it was told; it is silent on whether it was told the right thing, and reading it as broader
  assurance is what allowed fourteen months to pass.

## Common Pitfalls

⚠️ **Reading a green CI suite as project health.** Every gate passed on the day this project was
switched off. Gates certify implementation, never intent.

⚠️ **Defining the training population with a join and never stating it.** An inner join to
payments silently excluded 567 never-paid invoices — the worst outcomes in the book — from
training and from every reported metric, and the test set inherited the same filter, so no
evaluation could reveal it.

**Accepting a large offline improvement without asking what changed.** Leakage lifted precision to
0.7635 from 0.4499 while leaving production at 0.2956, worse than the clean model. An
implausibly large gain is a prompt to inspect the feature's timing, not to celebrate.

**Importing warnings without measuring them.** Random splitting on temporal data is a real hazard
and cost effectively nothing here — 0.5232 against 0.5280 on identical test rows — because the
drift is in the base rate rather than the relationship. Rituals adopted without measurement crowd
out the checks that would have mattered.

**Choosing the label threshold for statistical convenience.** Seven days was learnable and thirty
days is operational. Where those differ, the operational number wins or the difference gets
written down and justified.

**Leaving the retrain decision to a meeting.** The drift window trips a calibration gap of 0.123
against a threshold of 0.05 and shows the rule overtaking the model; expressed as a function it
pages on the day, expressed as a recurring agenda item it takes six weeks.

**Adding gates in the order a checklist lists them.** Every check here exists because a specific
earlier incident demanded it. A suite grown from real failures is maintained; a suite imported
wholesale is bypassed.

## Interview Questions

1. **Derive this.** Define detection distance and explain why it, rather than defect count,
   predicts a project's cost of failure. *Answer shape:* it is the gap between the stage that
   introduces a defect and the earliest gate able to observe it; rework cost grows with the number
   of downstream artifacts built on the flawed decision, so a defect caught at its own stage costs
   an afternoon while one with no offline detector costs however long until a business review
   notices — here, fourteen months.
2. **Design this.** A team hands you a model with a fully green CI suite and asks whether it should
   ship. What do you check that the suite cannot? *Answer shape:* the label spec against the
   operational trigger; the training population against the scoring population; the intervention's
   break-even precision against achievable precision; whether a business metric is pre-registered
   with a review date; and whether a baseline runs in parallel as a control.
3. **Debug this.** A model beats its baseline offline, reproduces exactly, is well calibrated, and
   produces no business effect over four quarters. Diagnose. *Answer shape:* eliminate
   implementation defects using the green suite itself, then interrogate framing — compare the
   label's definition to the operational decision it feeds, audit the population for a silent join
   filter, and check whether the intervention could pay at achievable precision. The absence of a
   bug is the diagnostic clue, not a dead end.
4. Why can a test set fail to reveal a survivorship-biased training population? *Answer shape:*
   the test set is drawn through the same join, so it inherits the identical filter; the metric is
   correct about the joined population and silent about the difference between that population and
   the served one. Only a population definition compared against production scoring volume detects
   it.
5. A leaked feature raises offline precision by more than thirty points. Why can it make production
   *worse* than not having the feature at all? *Answer shape:* the model allocates weight to the
   leaked column and correspondingly less to honest features; at serving time the column is
   constant, so the model is left relying on an under-weighted remainder — 0.2956 against a clean
   0.4499 here.
6. When is a random train/test split on time-ordered data actually harmful, and how would you find
   out for your problem? *Answer shape:* when the feature-outcome relationship changes over time,
   not merely the base rate; measure it by holding the test rows fixed and varying only whether
   training saw contemporaneous data — here that gave 0.5232 against 0.5280, so the hazard was
   negligible for this drift.
7. How do you decide when to add a gate? *Answer shape:* add one when a specific failure has
   occurred or when a decision has been written down that a check can reference; gates are cheap
   to run and not free to maintain, so a suite grown from real incidents survives while an imported
   checklist is bypassed. Note that some intent can only be gated once it is recorded — the
   artifact comes first, the check second.

## Key Takeaways

- The lifecycle is a dependency graph with rework costs, not a diagram; what matters is which
  stage a defect enters and which gate can see it.
- Gates certify that the pipeline did what you asked and are structurally incapable of certifying
  that you asked for the right thing.
- Five of seven injected defects were caught offline; the two that reached production were
  mis-specifications introduced during framing, and every gate stayed green for both.
- A decision that is not written down cannot be gated — the label spec and the scoring-population
  definition are what convert intent into something checkable.
- An inner join defines a training population silently: 567 never-paid invoices, the worst outcomes
  in the portfolio, were absent from every number this series reported.
- Leakage does not merely fail to help; at 0.2956 against a clean 0.4499 it left production worse
  than the feature's absence, because it displaced honest signal.
- Measure inherited warnings before adopting them — random splitting cost 0.5232 against 0.5280
  here, because this drift moves the base rate rather than the relationship.
- Terminate monitoring in a function, not a meeting: a calibration gap of 0.123 against a 0.05
  threshold pages on the day rather than after a quarter.

## Related

**Backward — the whole of series 01 assembles here**

- **01.1 Rules or Learning?** — supplied the baseline gate, the capacity, the calibration monitor
  and the drift whose retrain decision closes this notebook.
- **01.2 First Contact with the PayFlow Data Universe** — supplied the data contract gate, and the
  inner join whose silent population filter is one of the two defects no gate catches.
- **01.3 Reproducibility as an Engineering Contract** — supplied the manifest and the noise band,
  and the config block where the two missing specs belong.
- **01.4 The Taxonomy of Learning Problems** — supplied the serving contract gate and the framing
  vocabulary that makes "wrong horizon" nameable.
- **01.5 Problem Framing** — supplied the label spec whose absence is the other invisible defect,
  and the break-even test that belongs beside any funding decision.

**Forward**

- **02.1 NumPy & Vectorized Computing** — the next series begins the toolkit that everything above
  was built on, at the depth the rest of the track requires.
- **09.1 Data Cleaning & Pre-processing** — the contracted ingestion layer this notebook gates.
- **12.1 Model Evaluation & Validation** — canonical home for population definition as part of
  validation, the leakage taxonomy, and temporal cross-validation; the two invisible defects here
  are its subject matter.
- **15.1 Logistic Regression & Classifier Practice** — thresholds, calibration and cost matrices
  made canonical, including recalibration after a label change.
- **33.1 Interpretability, Fairness & Governance** — who is eligible to be scored is a governance
  question as much as a modelling one.
- **34.1 ML in Production** — the gate suite, artifact registry, staged rollout and retraining
  loop built properly, on the reader's own stack.
- **35.1 Capstones & Interview Synthesis** — where an end-to-end project is run with these gates
  in place from the first day rather than added after an incident.